# Predicting Peak Electricity Demand in Massachusetts
## Part 2 of 3: Exploratory Data Analysis

This notebook loads the cleaned, combined dataset produced in Part 1 and explores its
temporal structure: daily/weekly/seasonal patterns in demand, the demand-temperature
relationship, and renewable generation patterns. These visuals both validate that the
data behaves like real electricity demand and motivate the feature engineering choices
made in Part 3.

## 2.1 Load the Cleaned Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_combined = pd.read_csv("isone_full_dataset.csv", index_col="period", parse_dates=True)
target = "Demand (MW)"

print(df_combined.shape)
df_combined.head()

## 2.2 Autocorrelation of Demand

Tests whether demand has "memory" of its own recent past — a real time series should show
a spike around lag 24 (the daily cycle).

In [ ]:
from pandas.plotting import autocorrelation_plot

autocorrelation_plot(df_combined[target].iloc[:2000])
plt.title("Autocorrelation of ISO-NE Hourly Electricity Demand (First 2000 Hours)")
plt.xlabel("Lag (Number of Hours Between Observations)")
plt.ylabel("Autocorrelation Coefficient (-1 to 1)")
plt.grid(alpha=0.3)
plt.show()

## 2.3 Full Time Series with Extreme Peaks Highlighted

In [ ]:
threshold = df_combined[target].quantile(0.95)
peaks = df_combined[df_combined[target] > threshold]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_combined.index, df_combined[target], color="steelblue", linewidth=0.6, label="Hourly Demand")
ax.scatter(peaks.index, peaks[target], color="red", s=8, label=f"Top 5% Peak Hours (>{threshold:.0f} MW)")
ax.set_title("ISO-NE Hourly Electricity Demand, 2021–2023, with Extreme Peaks Highlighted")
ax.set_xlabel("Date")
ax.set_ylabel("Demand (MW)")
ax.legend()
plt.show()

## 2.4 Demand Heatmap: Hour of Day × Month

In [ ]:
pivot = df_combined.pivot_table(values=target, index="hour", columns="month", aggfunc="mean")

def hour_to_label(h):
    if h == 0:
        return "12 AM"
    elif h < 12:
        return f"{h} AM"
    elif h == 12:
        return "12 PM"
    else:
        return f"{h - 12} PM"

hour_labels = [hour_to_label(h) for h in pivot.index]

plt.figure(figsize=(10, 10))
ax = sns.heatmap(pivot, cmap="YlOrRd", cbar_kws={"label": "Avg Demand (MW)"})
ax.set_yticklabels(hour_labels, rotation=0)
plt.title("Average Electricity Demand by Time of Day and Month")
plt.xlabel("Month (1 = January, 12 = December)")
plt.ylabel("Time of Day")
plt.show()

## 2.5 Weekday vs. Weekend Demand Profile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for is_weekend, label in [(0, "Weekday"), (1, "Weekend")]:
    subset = df_combined[df_combined["is_weekend"] == is_weekend]
    hourly = subset.groupby("hour")[target].mean()
    ax.plot(hourly.index, hourly.values, marker="o", label=label)

ax.set_title("Average Demand by Hour: Weekday vs Weekend")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Average Demand (MW)")
ax.legend()
plt.grid(alpha=0.3)
plt.show()

## 2.6 Monthly Demand Distribution

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x="month", y=target, data=df_combined)
plt.title("Distribution of Hourly Demand by Month (Shows Variability, Not Just Averages)")
plt.xlabel("Month")
plt.ylabel("Demand (MW)")
plt.show()

## 2.7 Year-over-Year Comparison

In [ ]:
df_combined["year"] = df_combined.index.year

fig, ax = plt.subplots(figsize=(10, 5))
for year in sorted(df_combined["year"].unique()):
    yearly = df_combined[df_combined["year"] == year].groupby("month")[target].mean()
    ax.plot(yearly.index, yearly.values, marker="o", label=str(year))

ax.set_title("Average Monthly Demand, Compared Year Over Year")
ax.set_xlabel("Month")
ax.set_ylabel("Average Demand (MW)")
ax.legend(title="Year")
plt.grid(alpha=0.3)
plt.show()

## 2.8 Demand vs. Temperature

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_combined["Temperature (°C)"], df_combined[target], alpha=0.1, s=5)
plt.title("Electricity Demand vs. Temperature (Boston, MA, 2021–2023)")
plt.xlabel("Temperature (°C)")
plt.ylabel("Demand (MW)")
plt.grid(alpha=0.3)
plt.show()

## 2.9 Solar Generation Profile

Confirms solar output follows a physically sensible daily cycle — near zero overnight,
peaking around midday.

In [ ]:
hourly_solar = df_combined.groupby("hour")["Solar Generation (MWh)"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly_solar.index, hourly_solar.values, marker="o", color="orange")
ax.set_title("Average Solar Generation by Hour of Day (ISO-NE, Local Time)")
ax.set_xlabel("Hour of Day (0 = Midnight, 12 = Noon, 23 = 11 PM)")
ax.set_ylabel("Average Solar Generation (MWh)")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)
plt.show()

## 2.10 Key EDA Takeaways

- Demand shows strong autocorrelation at daily (24h) and weekly (168h) lags — a genuine
  time series with real temporal structure, unlike the discarded synthetic Kaggle dataset.
- Demand follows a clear seasonal pattern: winter heating and summer cooling both drive
  elevated demand, with a trough in the mild spring/fall shoulder months.
- Weekday and weekend demand profiles differ meaningfully, motivating the `is_weekend`
  feature used in modeling.
- Solar generation follows the expected midday-peak, near-zero-overnight daily cycle,
  confirming the renewable generation data is correctly time-aligned.

These patterns motivate the lag features, degree-day features, and calendar features
built in Part 3.